# 02 - Noise robustness

How well does MPSE reconstruct a shape as the input distance matrices get
noisier? Here we sweep the amount of **distance noise** and track the Chamfer
error of both MPSE and the plain-MDS baseline.

See `mpe3d.noise` for the noise models (`distance` and `matching`).

In [ ]:
import numpy as np
import plotly.graph_objs as go

from mpe3d import datasets, reconstruct

## Base shape

We reuse the demo torus. `noise_amount` is the fraction of points perturbed and
`noise_level` scales the magnitude relative to each matrix's value range.

In [ ]:
points = datasets.get_dataset_points("demo:torus", n_points=250, normalize=True)

noise_levels = [0.0, 0.02, 0.05, 0.1, 0.2]
noise_amount = 0.5  # perturb half of the points at each level

## Sweep

For each noise level we run a fresh reconstruction (fixed seed for
reproducibility) and record the MPSE and baseline Chamfer distances.

In [ ]:
mpse_chamfer = []
baseline_chamfer = []

for level in noise_levels:
    result = reconstruct(
        points,
        n_perspectives=5,
        points_in_at_least=3,
        noise_type="distance",
        noise_amount=noise_amount,
        noise_level=level,
        batch_size=64,
        max_iter=120,
        verbose=0,
        rng=np.random.default_rng(0),
    )
    mpse_chamfer.append(result.chamfer)
    baseline_chamfer.append(result.baseline["chamfer"])
    print(f"noise_level={level:>4}: MPSE={result.chamfer:8.3f}  "
          f"baseline={result.baseline['chamfer']:8.3f}")

## Results

Chamfer distance vs. noise level (lower is better).

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=noise_levels, y=mpse_chamfer, mode="lines+markers",
                         name="MPSE"))
fig.add_trace(go.Scatter(x=noise_levels, y=baseline_chamfer, mode="lines+markers",
                         name="MDS baseline"))
fig.update_layout(title="Reconstruction error vs. distance noise",
                  xaxis_title="noise level", yaxis_title="Chamfer distance",
                  width=650, height=450)
fig